In [1]:
import pandas as pd
import altair as alt

In [2]:
data = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"

df = pd.read_csv(data)
df

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8857,Department of Transportation,Belvidere Maintenance Storage Facility - Boone...,9797 Illinois Rte. 76,Belvidere,61008,Boone,16,Adam Kinzinger,69,Sosnowski Joe,...,In Use,0,0,432,1,0,0,Storage,NaN,NaN
8858,Department of Transportation,Belvidere Maintenance Storage Facility - Boone...,9797 Illinois Rte 76,Belvidere,61008,Boone,16,Adam Kinzinger,69,Sosnowski Joe,...,In Use,0,0,330,1,0,0,Storage,NaN,NaN
8859,Department of Transportation,Quincy Maintenance Storage Facility,800 Koch's Lane,Quincy,62305,Adams,18,Darin M. LaHood,94,Frese Randy E.,...,In Use,0,1987,130,1,0,0,Storage,High Hazard,NaN
8860,Illinois Community College Board,Illinois Valley Community College - Oglesby,815 North Orlando Smith Avenue,Oglesby,61348,LaSalle,16,Adam Kinzinger,76,Long Jerry Lee,...,In Use,1971,1971,49552,1,1,0,Education,Education,Not provided


In [3]:
df.isna().sum()

Agency Name                  0
Location Name                0
Address                     51
City                         0
Zip code                     0
County                      25
Congress Dist                0
Congressional Full Name    163
Rep Dist                     0
Rep Full Name               23
Senate Dist                  0
Senator Full Name           23
Bldg Status                  0
Year Acquired                0
Year Constructed             0
Square Footage               0
Total Floors                 0
Floors Above Grade           0
Floors Below Grade           0
Usage Description            0
Usage Description 2         30
Usage Description 3         88
dtype: int64

In [7]:
df = df[df['Square Footage'] > 0]
df['Decade'] = (df['Year Constructed'] // 10 * 10).astype(int)
df_usage = (df.groupby('Usage Description')['Square Footage'].mean().reset_index().rename(columns={'Square Footage': 'Mean Square Footage'}))
df_usage

,Usage Description,Mean Square Footage
0,Assembly,25481.137405
1,Business,26148.312680
2,Detention & Correc,12402.081267
3,Education,52874.031128
4,Health Care,22114.144737
5,Industrial,9277.145359
6,Mercantile,4168.189189
7,Not provided,384.000000
8,Public,560.000000
9,Residential,8295.705414


In [8]:
plot1 = alt.Chart(df_usage).mark_bar().encode(
    x=alt.X('Mean Square Footage:Q', title='Average Square Footage'),
    y=alt.Y('Usage Description:N', sort='-x', title='Usage Type'),
    tooltip=['Usage Description:N', alt.Tooltip('Mean Square Footage:Q', format=',.0f')]
).properties(title='Average Building Size by Usage Type')

plot1

alt.Chart(...)

**Plot 1 Write-up**

The visualization shows the average building size by square footage by usage type. Each bar is encoded to a usage type while the length of the bar is encoded to the average square footage of the building (quantitative). I sorted the bars in descending order so we can easily tell which usage types are the largest to the smallest.

The data transformation was dropping rows with 0 or lower values for square footage. I didn't need to drop null values because the column didn't have any. I then grouped the buildings by usage type and graphed it out with a simple bar chart. I didn't need to use a color map, as this was a simple bar chart, which all measured the same thing, so there was nothing I needed to distinguish from each other.

In [9]:
df = df.dropna(subset=['Year Acquired'])
df = df[(df['Year Acquired'] > 0) & (df['Year Acquired'] <= 2026)]
df_acquired = (df.groupby('Year Acquired').size().reset_index(name='Number of Buildings'))
df_acquired

,Year Acquired,Number of Buildings
0,1802,2
1,1810,3
2,1832,1
3,1837,1
4,1838,1
...,...,...
165,2015,20
166,2016,10
167,2017,1
168,2018,4


In [10]:
plot2 = alt.Chart(df_acquired).mark_bar().encode(
    x=alt.X('Year Acquired:O', title='Year Acquired', axis=alt.Axis(labelAngle=-45, values=list(range(1900, 2026, 10)))),
    y=alt.Y('Number of Buildings:Q', title='Number of Buildings Acquired'),
    tooltip=['Year Acquired:O', 'Number of Buildings:Q']
).properties(
    title='Number of State Buildings Acquired Per Year', width=600) #graph is too long without width
plot2

alt.Chart(...)

**Plot 2 Write-up**

This visualization shows plots out the number of buildings acquired by Illinois state vs the year. I encoded the year to the x-axis as an ordinal variable and I encoded the number of buildings acquired to the y-axis as a quantitative variable. Using a bar chart is ideal here, as it allows us to easily visualize the trends and standout years of building acquisition. I added an interactive element to hover each bar and know the eact number of buildings acquired and year as we have over a hundred years worth of bars, which makes it difficult to distinguish.

One transformation I had to make to the data was to drop all the rows with a 0 in Year Acquired, as its likely a stand in for unknown. I want to ensure that my graph did not potentially take these rows into account. Otherwise, since the other variables I needed were accounted for I did no other transformations. As for the color scheme of the graph, I went for the default color, as there is no need to distinguish anything particular here.